# Tutorial 5 — Structural LTN

Tutorials 1–4 had one kind of index: a **variable** ranges over a batch of independent individuals, and a formula is evaluated on each of them (or on each combination). Many data have a second kind of index, *inside* one individual: the days of a hospital stay, the frames of a video, the nodes of a graph. sLTN calls these **structural dimensions**.

The distinction matters for the logic. Two variables are independent, so `Fever(x) & Fever(y)` is evaluated on every pair of patients. The days of a stay are not independent. "Day 3" means day 3 *of the same patient*, the days are ordered, and the relevant statements relate days to each other, e.g. "the fever persists to the next day" or "once recovered, the patient has no fever again".

| Concept | Declared with | Role |
|---|---|---|
| structural dimension | `sig.dimension("Day")` | an index set inside each individual; its size comes from the data |
| structural variable | `sig.structural_variable("d", "Day")` | names a position along `Day`; the only way to refer to a dimension in a formula |
| structural relation | `sig.structural_relation("next", ["Day", "Day"])` | a fixed [0, 1] matrix over positions, used as a guard or as an atom |
| consumed / produced dims | `input_dims=`, `output_dims=` | which structural axes a predicate or function receives whole, and which it returns |

The running example is a hospital ward in which the temperature of each patient is measured once a day. Sections 1–6 declare the signature and the data and write formulas about single days, whole stays and pairs of days. Section 7 trains a network that recognises days with fever from a few labels and two rules on the course of the disease.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import sltn
from sltn import Signature, Interpretation, KB
from sltn.interpretation.operators import AggregPMean, AggregPMeanError, AggregMin, AggregMax

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1 — Signature: dimensions and structural variables

A **dimension** is declared by name only. It has no size in the signature, and its size is given by the tensors grounded over it.

A dimension never appears in a formula. Its positions are referred to through a **structural variable** declared over it, here `d`, which ranges over the days. `forall Day: ...` and `x[Day]` are therefore errors.

A variable or a constant declared with `dims=["Day"]` holds one value per day. Here `x` is a batch of patients, each with one reading per day.

In [ ]:
sig = Signature("ward")
sig.sort("Reading")                          # one temperature reading
sig.dimension("Day")
sig.structural_variable("d", "Day")          # d ranges over the days

sig.variable("x", "Reading", dims=["Day"])   # patients, one reading per day
print(sig)

**Data layout.** `x` is grounded with a tensor of shape `[patients, days, *shape of Reading]`, that is, the variable axis, then one axis per dimension in the order of `dims`, then the shape of the sort. The following cell grounds four patients over five days.

| patient | course |
|---|---|
| 0 | recovers |
| 1 | gets worse |
| 2 | never has fever |
| 3 | recovers, then relapses |

In [ ]:
temps = torch.tensor([
    [39.0, 38.6, 38.0, 37.2, 36.8],   # 0: recovers
    [37.0, 37.4, 38.2, 38.9, 39.3],   # 1: gets worse
    [36.8, 36.9, 36.7, 36.9, 36.8],   # 2: never has fever
    [38.6, 37.1, 36.9, 38.4, 38.8],   # 3: recovers, then relapses
])

interp = Interpretation(sig)
interp["Reading"] = sltn.Type("temperature", shape=(1,))
interp["x"] = temps.unsqueeze(-1)            # [4 patients, 5 days, 1]
print(interp("x[d]"))

`x[d]` is a term with three axes: a **variable** axis `x` (the patients), a **structural** axis `d` (the days) and a **domain** axis `temperature` (the reading).

## 2 — Naming axes

Every structural axis in a formula must be named with a structural variable, so `Fever(x)` is rejected.

In [ ]:
sig.predicate("Fever", ["Reading"])
interp["Fever"] = lambda v: torch.sigmoid(3 * (v - 37.8))

try:
    sig.parse("Fever(x)")
except ValueError as e:
    print(f"Fever(x)\n  -> {e}\n")

The name of an axis determines its meaning, and **the same name denotes the same day**. Two days are referred to at once with an alias of `d`, written `d1`, `d2` or `d'`, which ranges over the same dimension and names a new axis.

- The denotation of `Fever(x[d]) & Fever(x[d])` has axes `[x, d]`, and each day is combined with itself.
- The denotation of `Fever(x[d]) & Fever(x[d1])` has axes `[x, d, d1]`, and every pair of days of the same patient is combined.

`free_structural_variables` lists the structural variables that are free in a parsed formula. It is computed from the formula alone, without computing its denotation.

In [ ]:
for s in ["Fever(x[d]) & Fever(x[d])", "Fever(x[d]) & Fever(x[d1])"]:
    phi = sig.parse(s)
    r = interp(phi)
    print(f"{s:30} free structural variables: {phi.free_structural_variables}   axes: {r.axis_names}  shape: {tuple(r.value.shape)}")

The patient axis `x` is also shared. A structural axis indexes positions *inside* each individual, so `d1` is another day of the same patient, never a day of another patient. Pairs of patients require an alias of the individual variable, `x1`.

## 3 — Three kinds of predicates

`input_dims` and `output_dims` decide what a predicate's grounding receives and returns. This is the main design choice in a structural signature.

| Declaration | The grounding receives | It returns | In a formula | Use it for |
|---|---|---|---|---|
| no dims | one reading: `[B, 1]` | `[B, 1]` | `Fever(x[d])`: `d` stays free | a property of one day |
| `input_dims=["Day"]` | the whole stay: `[B, Day, 1]` | `[B, 1]` | `Recovered(x[d])`: `d` is **consumed** | a property of the whole stay |
| `input_dims=["Day"], output_dims=["Day"]` | the whole stay: `[B, Day, 1]` | `[B, Day, 1]` | `Rising(x[d])[d]`: the output axis is named | a property of one day that depends on the other days |

- **No dims.** sLTN flattens patients and days into one batch, `B = 4 × 5 = 20`, and the grounding receives one reading per row. It therefore cannot use the other days.
- **Consumed.** The grounding receives the whole sequence of each patient, `B = 4`, and returns one value per patient. The axis `d` is absent from the denotation.
- **Consumed and produced.** The grounding receives the whole sequence and returns one value per day, as a recurrent or convolutional network does. The output axis is named after the call, `Rising(x[d])[d]`.

`Rising` compares each reading with the reading of the previous day. Day 0 has no previous day, so it is compared with itself, which gives 0.5. The groundings below print the shape they receive.

In [ ]:
sig.predicate("Recovered", ["Reading"], input_dims=["Day"])
sig.predicate("Rising", ["Reading"], input_dims=["Day"], output_dims=["Day"])


def fever(v):
    print("  Fever receives    ", tuple(v.shape))
    return torch.sigmoid(3 * (v - 37.8))


def recovered(v):                                # v: [B, Day, 1]
    print("  Recovered receives", tuple(v.shape))
    return torch.sigmoid(3 * (37.5 - v[:, -1]))  # last reading below 37.5: [B, 1]


def rising(v):                                   # v: [B, Day, 1]
    print("  Rising receives   ", tuple(v.shape))
    prev = torch.cat([v[:, :1], v[:, :-1]], dim=1)
    return torch.sigmoid(3 * (v - prev))         # [B, Day, 1]


interp["Fever"] = fever
interp["Recovered"] = recovered
interp["Rising"] = rising

for s in ["Fever(x[d])", "Recovered(x[d])", "Rising(x[d])[d]"]:
    print(s)
    r = interp(s)
    print(f"  result axes {r.axis_names}, shape {tuple(r.value.shape)}\n")

`Fever` and `Rising` both give one truth degree per patient and day, but only `Rising` can depend on the other days. The declaration therefore determines which readings the grounding receives, not the shape of its denotation.

A predicate that consumes `Day` must receive an argument whose `Day` axis is named, so `Recovered(x)` is an error. A predicate that produces `Day` must name its output axis, so `Rising(x[d])` is an error as well.

In [ ]:
interp["Fever"] = lambda v: torch.sigmoid(3 * (v - 37.8))     # the same groundings, without printing the shapes
interp["Recovered"] = lambda v: torch.sigmoid(3 * (37.5 - v[:, -1]))
interp["Rising"] = lambda v: torch.sigmoid(3 * (v - torch.cat([v[:, :1], v[:, :-1]], dim=1)))

try:
    sig.parse("Rising(x[d])")
except ValueError as e:
    print("Rising(x[d])\n  ->", e)

**Naming the output axis.** The name after the call can be any structural variable over `Day`. `Rising(x[d])[d]` reuses the name of the input axis, so the output day is the same day as `d` elsewhere in the formula. A new name, as in `Rising(x[d])[d1]`, gives the output an axis of its own.

In [ ]:
for s in ["Fever(x[d]) & Rising(x[d])[d]", "Fever(x[d]) & Rising(x[d])[d1]"]:
    r = interp(s)
    print(f"{s:35} axes {r.axis_names}")

The first formula reads "fever on day `d` and rising on day `d`", the second "fever on day `d` and rising on day `d1`", for every pair of days.

**Functions** are declared with the same `input_dims` and `output_dims`. A smoothing function consumes and produces `Day`, and returns a sequence of the same length. The mean temperature of a stay consumes `Day`. Their denotations are terms, and they are used as arguments of predicates.

In [ ]:
sig.function("smooth", ["Reading"], "Reading", input_dims=["Day"], output_dims=["Day"])
sig.function("mean_temp", ["Reading"], "Reading", input_dims=["Day"])

def moving_average(v):                           # v: [B, Day, 1], 3-day window
    padded = F.pad(v.transpose(1, 2), (1, 1), mode="replicate")
    return F.avg_pool1d(padded, 3, stride=1).transpose(1, 2)

interp["smooth"] = moving_average
interp["mean_temp"] = lambda v: v.mean(dim=1)    # [B, 1]

print(interp("smooth(x[d])[d]").axis_names)
print(interp("Fever(smooth(x[d])[d])").value.squeeze(-1))
print(interp("Fever(mean_temp(x[d]))").value.squeeze(-1))

## 4 — Quantifying over days

`∀ d: φ` and `∃ d: φ` aggregate over the axis `d` and remove it. They are written as first-order quantifiers, and sLTN recognises from the variable `d` that the aggregation is over a structural axis.

**Each dimension has its own operators**, stored under the keys `"∀,Day"` and `"∃,Day"`. All structural variables over `Day` (`d`, `d1`, …) use these entries. A dimension without its own operators uses the current first-order `∀` and `∃`, that is, `AggregPMeanError(p=2)` and `AggregPMean(p=2)` by default, or `AggregMin` and `AggregMax` under `Logic.classical()`. Changing `logic["∀"]` therefore also changes `∀ d` until `logic["∀,Day"]` is set explicitly.

In [ ]:
print("∀,Day :", interp.logic["∀,Day"])
print("∃,Day :", interp.logic["∃,Day"])

In [ ]:
always = interp("∀ d: Fever(x[d])").value.squeeze(-1)
some_day = interp("∃ d: Fever(x[d])").value.squeeze(-1)
for k in range(4):
    print(f"patient {k}: fever every day {always[k]:.2f}   fever on some day {some_day[k]:.2f}")

`x` is still free, so the denotation contains one value per patient. With min and max the values are closer to 0 and 1, since "every day" is decided by the coldest day and "some day" by the hottest. Each value, and therefore its gradient, then depends on a single day.

In [ ]:
interp.logic["∀,Day"] = AggregMin()
interp.logic["∃,Day"] = AggregMax()
print("min:", interp("∀ d: Fever(x[d])").value.squeeze(-1))
print("max:", interp("∃ d: Fever(x[d])").value.squeeze(-1))

interp.logic["∀,Day"] = AggregPMeanError(p=2)
interp.logic["∃,Day"] = AggregPMean(p=2)

**Closed formulas.** A quantifier binds either individual variables or structural variables, not both, so `∀ x, d: ...` is an error. The two quantifiers are nested instead.

- `∀ x: ∃ d: Fever(x[d])` reads "every patient has fever on some day".
- `∃ d: ∀ x: Fever(x[d])` reads "there is a day on which every patient has fever".

In [ ]:
try:
    sig.parse("∀ x, d: Fever(x[d])")
except ValueError as e:
    print("∀ x, d: ...\n  ->", e, "\n")

for s in ["∀ x: ∃ d: Fever(x[d])", "∃ d: ∀ x: Fever(x[d])"]:
    print(f"{s:25} {interp(s).value.item():.3f}")

## 5 — Structural relations

A **structural relation** is grounded with a fixed tensor of values in [0, 1] over positions, never with a function. In the following cell `next(d, d1)` holds when `d1` is the day after `d`, and `before(d, d1)` holds when `d1` is later than `d`.

In [ ]:
sig.structural_relation("next", ["Day", "Day"])
sig.structural_relation("before", ["Day", "Day"])

days = torch.arange(5)
interp["next"] = (days.unsqueeze(1) + 1 == days.unsqueeze(0)).float()   # [5, 5]
interp["before"] = (days.unsqueeze(1) < days.unsqueeze(0)).float()
print(interp["next"])
print(interp["before"])

### As a guard

A relation after `|` restricts the quantifier to the positions at which the relation holds, as a guard does in Tutorial 2. "The fever persists to the next day" is written

```
∀ d, d1 | next(d, d1): (Fever(x[d]) -> Fever(x[d1]))
```

The quantifier aggregates only the 4 pairs of consecutive days and excludes the other 21 pairs.

This flat form aggregates over all consecutive pairs at once. The nested form `∀ d: ∀ d1 | next(d, d1): …` aggregates over the successors of each day first. On the last day, which has no successor, the inner guard is empty, so `∀` gives 1 (weak next) and `∃` gives 0 (strong next). With a relation of 0s and 1s the two forms differ only on that day. With soft weights, the flat form gives more weight to the days whose row of the relation has a larger total weight.

In [ ]:
persists = interp("∀ d, d1 | next(d, d1): (Fever(x[d]) -> Fever(x[d1]))").value.squeeze(-1)
for k in range(4):
    print(f"patient {k}: {persists[k]:.2f}")

Patients 0 and 3 have the lowest values, since their fever stops, and on that day `Fever(x[d]) -> Fever(x[d1])` is false. Patient 1, whose fever only increases, and patient 2, who never has fever, satisfy the formula.

"Once recovered, the patient has no fever again" relates each day to all later days, through `before`. Patient 3, who relapses, violates it, and so does patient 1, who has no fever at first and has fever later.

In [ ]:
no_relapse = interp("∀ d, d1 | before(d, d1): (¬Fever(x[d]) -> ¬Fever(x[d1]))").value.squeeze(-1)
for k in range(4):
    print(f"patient {k}: {no_relapse[k]:.2f}")

## 6 — Axis selection: picking one day

An **axis selection** `[d=k]` keeps position `k` of the structural axis `d` and removes the axis. Negative indices count from the end, as in Python. The axis selection goes after the atom that names the axis.

In [ ]:
admission = interp("Fever(x[d])[d=0]")
discharge = interp("Fever(x[d])[d=-1]")
print("axes:", admission.axis_names)
print("fever on admission:", admission.value.squeeze(-1))
print("fever at discharge:", discharge.value.squeeze(-1))

A formula with an axis selection combines with the others like any formula. "Admitted with fever, and recovered":

In [ ]:
print(interp("Fever(x[d])[d=0] & Recovered(x[d])").value.squeeze(-1))

try:
    interp("Fever(x[d])[d=7]")                   # only 5 days
except IndexError as e:
    print("\nFever(x[d])[d=7]\n  ->", e)

## 7 — Learning with structure

A ward of 200 patients is observed for 7 days. Every patient is admitted with fever, which lasts 1 to 5 days and does not return. The temperature is measured once a day, around 38.1 °C on a day with fever and around 36.9 °C on a day without, with a standard deviation of 0.5 °C. The two distributions overlap, so a single reading is often ambiguous. Only **5%** of the days are labelled, never the day of admission. `Fever` is trained from these labels, once alone and once with two rules on the course of the disease.

**Patients as a dimension.** Each label refers to one reading. "Patient 17 had fever on day 3" must be compared with `temp[17, 3]` only. The labels are therefore masks over (patient, day), and a structural relation ranges over dimensions only, not over individual variables. The patients are thus a second dimension, `Patient`, and the readings are a **constant** with dims `[Patient, Day]`, one tensor of shape `[200, 7, 1]`. In `Fever(temp[p, d])[d]` guarded by `Sick(p, d)`, the reading and the mask share the axes `p` and `d`, so each label is compared with its own reading only.

**Labels as masks.** `Sick` and `Healthy` are two relations over `[Patient, Day]`, used as guards. Each labelling clause then reads as a sentence, e.g. "on every day labelled sick, `Fever` holds", and the unlabelled days are excluded from the quantifier (section 5).

In [ ]:
torch.manual_seed(0)
N, T = 200, 7
last = torch.randint(1, 6, (N,))                                         # fever lasts 1..5 days
fever_true = (torch.arange(T).unsqueeze(0) < last.unsqueeze(1)).float()  # [N, T]
readings = 36.9 + 1.2 * fever_true + 0.5 * torch.randn(N, T)
labelled = torch.rand(N, T) < 0.05
labelled[:, 0] = False                                                   # admission day never labelled
print(f"{labelled.sum().item()} labelled days out of {N * T}")
print(f"accuracy of the threshold 37.5 on single readings: "
      f"{((readings > 37.5).float() == fever_true)[~labelled].float().mean():.3f}")

`Fever` consumes and produces `Day` (section 3). It is grounded with a bidirectional GRU that receives the whole stay and returns one truth degree per day, so it can use the neighbouring days to decide an ambiguous reading. `p` is not consumed, so the 200 patients form the batch.

In [ ]:
ward = Signature("ward_learning")
ward.sort("Reading")
ward.dimension("Patient")
ward.structural_variable("p", "Patient")
ward.dimension("Day")
ward.structural_variable("d", "Day")

ward.constant("temp", "Reading", dims=["Patient", "Day"])
ward.predicate("Fever", ["Reading"], input_dims=["Day"], output_dims=["Day"])
ward.structural_relation("Sick", ["Patient", "Day"])
ward.structural_relation("Healthy", ["Patient", "Day"])
ward.structural_relation("next", ["Day", "Day"])


class FeverNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.gru = nn.GRU(1, 16, batch_first=True, bidirectional=True)
        self.out = nn.Linear(32, 1)

    def forward(self, v):                    # v: [B, Day, 1]
        h, _ = self.gru(v - 37.5)
        return torch.sigmoid(self.out(h))    # [B, Day, 1]


def make_interp():
    i = Interpretation(ward)
    i.to(device)
    i["Reading"] = sltn.Type("temperature", shape=(1,))
    i["temp"] = readings.unsqueeze(-1)
    i["Sick"] = (labelled & (fever_true == 1)).float()
    i["Healthy"] = (labelled & (fever_true == 0)).float()
    days = torch.arange(T)
    i["next"] = (days.unsqueeze(1) + 1 == days.unsqueeze(0)).float()
    i["Fever"] = FeverNet()
    i.logic["∀,Patient"] = AggregPMeanError(p=4)
    i.logic["∀,Day"] = AggregPMeanError(p=4)
    return i

The knowledge base contains four clauses. The first two are the labels. The other two state properties of the disease.

- Every patient is admitted with fever, i.e. `Fever` holds on day 0, written with an axis selection.
- The fever does not return, i.e. if there is fever on the next day, there is fever on the current day, written with a guard on `next`.

Both dimensions use pMeanError with p = 4, which is closer to the minimum than p = 2. A clause is then satisfied only if it holds for almost every patient and day, not only on average.

In [ ]:
labels = [
    ward.parse("∀ p: ∀ d | Sick(p, d): Fever(temp[p, d])[d]"),
    ward.parse("∀ p: ∀ d | Healthy(p, d): ¬Fever(temp[p, d])[d]"),
]
knowledge = [
    ward.parse("∀ p: Fever(temp[p, d])[d][d=0]"),
    ward.parse("∀ p: ∀ d, d1 | next(d, d1): (Fever(temp[p, d1])[d1] -> Fever(temp[p, d])[d])"),
]


def train(clauses, epochs=300):
    torch.manual_seed(100)
    i = make_interp()
    kb = KB(clauses)
    optimizer = torch.optim.Adam(i.parameters(), lr=1e-2)
    for _ in range(epochs):
        optimizer.zero_grad()
        loss = 1 - i(kb).value.squeeze()
        loss.backward()
        optimizer.step()
    return i


def accuracy_unlabelled(i):
    with torch.no_grad():
        r = i("Fever(temp[p, d])[d]")
    predicted = (r.value.squeeze(-1).cpu() > 0.5).float()
    return (predicted == fever_true)[~labelled].float().mean().item()


only_labels = train(labels)
with_knowledge = train(labels + knowledge)
print(f"accuracy on the unlabelled days, labels only:      {accuracy_unlabelled(only_labels):.3f}")
print(f"accuracy on the unlabelled days, labels + knowledge: {accuracy_unlabelled(with_knowledge):.3f}")

Both networks are more accurate than the threshold on single readings, because they receive the whole stay. The two rules on the course of the disease increase the accuracy further. They require no labels, since they hold for every patient, so they also constrain the 95% of the days that are unlabelled.

## Summary

- A **dimension** is an index set inside each individual (days, frames, nodes). Its size comes from the data and must agree across every tensor over it.
- A dimension is referred to only through a **structural variable** (`d` over `Day`). Every structural axis is named: `x[d]`, never `x` or `x[Day]`. The same name means the same position; an alias (`d1`) names a second one.
- `input_dims` / `output_dims` decide what a grounding sees: no dims = one position at a time; consumed = the whole sequence, one value back; consumed and produced = the whole sequence, one value per position (`Rising(x[d])[d]`).
- `∀ d` and `∃ d` use the operators of their dimension, `logic["∀,Day"]` and `logic["∃,Day"]`, which default to the current `logic["∀"]` and `logic["∃"]`. A quantifier binds variables of one kind and one dimension, and the others are nested.
- A **structural relation** is a fixed [0, 1] tensor. As a guard it restricts the positions a quantifier aggregates.
- An axis selection `[d=k]` keeps one position of a structural axis and removes the axis; negative indices count from the end.
- Read results by axis name (`r.axis_names`): their axes are in alphabetical order within each role.
- A structural relation ranges over dimensions only: to mask (patient, day) pairs, the patients must be a dimension. Data and labels carried by two variables are paired with `∀ (x, y)` (Tutorial 2), not crossed.

---

**Next:** [Tutorial 6 — Advanced Features](6-AdvancedFeatures.ipynb) covers soft and unary relations, relations as atoms, structural variables as terms, several dimensions, infix symbols, the evaluation cache, multi-objective training, PyTorch Lightning, crisp Boolean evaluation and the checks on truth values.